# Example 3 — Wine Quality Regression

Train a regression model to predict numeric wine-quality scores. No GPU is required.

In [ ]:
%pip -q install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
print("Setup complete!")

## 1. Load the dataset
The UCI red wine dataset contains chemical measurements and a numeric quality score.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")
print("Shape:", df.shape)
display(df.head())

## 2. Explore the data

In [ ]:
print("Missing values:")
print(df.isna().sum())
display(df.describe())

In [ ]:
sns.histplot(df["quality"], discrete=True)
plt.title("Wine quality score distribution")
plt.xlabel("Quality score")
plt.show()

## 3. Define features and target
This is **regression** because the target is a numeric score, not a category.

In [ ]:
X = df.drop(columns=["quality"])
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))

## 4. Train a Random Forest Regressor

In [ ]:
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", RandomForestRegressor(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ))
])
model.fit(X_train, y_train)
print("Training complete!")

## 5. Evaluate regression performance
MAE and RMSE are errors in quality-score units; lower is better. R² compares performance to a simple mean baseline and should be interpreted in context.

In [ ]:
pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, pred, alpha=0.65)
plt.xlabel("Actual quality")
plt.ylabel("Predicted quality")
plt.title("Actual vs predicted wine quality")
low = min(y_test.min(), pred.min())
high = max(y_test.max(), pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.tight_layout()
plt.show()

## 6. Predict and compare one example

In [ ]:
sample = X_test.iloc[[0]]
prediction = float(model.predict(sample)[0])
print("Predicted quality:", round(prediction, 2))
print("Actual quality:", int(y_test.iloc[0]))

## 7. Save the model

In [ ]:
joblib.dump(model, "wine_quality_regressor.joblib")
print("Saved wine_quality_regressor.joblib")

## Try it yourself
1. Train a `LinearRegression` model and compare MAE/RMSE.
2. Explain why we use regression metrics instead of accuracy.
3. Investigate which features are most useful.
4. Why might rare quality scores be harder to predict?